In [1]:
import os 
import numpy as np 

In [2]:

train_data = np.memmap("data/train.bin", dtype=np.uint16, mode="r")
val_data   = np.memmap("data/val.bin",   dtype=np.uint16, mode="r")

In [3]:
import torch
block_size = 8
batch_size = 32
vocab_size = 4096
max_iters = 3000
eval_interval = 300
learning_rate = 1e-2
device = 'cuda' if torch.cuda.is_available() else 'cpu'
eval_iters = 200
n_embd = 32
def get_batch(split):
    data = train_data if split == "train" else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([torch.from_numpy(data[i : i + block_size].astype(np.int64))         for i in ix.tolist()])
    y = torch.stack([torch.from_numpy(data[i + 1 : i + 1 + block_size].astype(np.int64)) for i in ix.tolist()])
    return x.to(device), y.to(device)

In [ ]:
import torch.nn.functional as F
from torch import nn 

class GPTLanguageModel(nn.Module):
  def __init__(self):
    super().__init__()
    self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
    self.positional_embedding_table = nn.Embedding(block_size, n_embd)
    self.sa_heads = MultiHeadAttention(4 , n_embd//4)
    self.lm_head = nn.Linear(n_embd, vocab_size)
  def forward(self, idx, targets=None):
          B, T = idx.shape  # (B, T)
          tok_emb = self.token_embedding_table(idx) # (B,T, N_EMBD)
          pos_emb = self.positional_embedding_table(torch.arange(T, device = device)) # T,N_EMBD
          x = tok_emb + pos_emb
          x = self.sa_heads(x)
          logits = self.lm_head(x) # (B,T,vocab_size)

          loss = None
          if targets is not None:
              B, T, V = logits.shape
              loss = F.cross_entropy(logits.view(B*T, V), targets.view(B*T))
          return logits, loss
  @torch.no_grad()
  def generate(self, idx, max_new_tokens):
          # idx: (B, T) tensor of token ids to start from
          for _ in range(max_new_tokens):
              idx_cond = idx[:, -block_size:]
              logits, _ = self(idx_cond)
              logits = logits[:, -1, :]
              probs = F.softmax(logits, dim=-1)
              next_id = torch.multinomial(probs, 1)
              idx = torch.cat([idx, next_id], dim=1)
          return idx

In [5]:
class Head(nn.Module):
    def __init__(self, head_size):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias = False)
        self.query = nn.Linear(n_embd, head_size, bias = False)
        self.value= nn.Linear(n_embd, head_size, bias = False)
        self.register_buffer('tril', torch.tril(torch.ones(block_size,block_size)))
    def forward(self,x):
        B,T,C = x.shape
        k = self.key(x)
        q = self.query(x)
        v = self.value(x)

        wei = q @ k.transpose(-2,-1) * k.shape[-1] ** -0.5
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf'))
        wei = F.softmax(wei, dim = -1)
        out = wei @ v
        return out

In [6]:
class MultiHeadAttention(nn.Module):
    """Multiple Attention Heads in parallel"""
    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList(Head(head_size) for _ in range(num_heads))

    def forward(self,x):
        return torch.cat([h(x) for h in self.heads], dim = -1)

In [ ]:
model = GPTLanguageModel().to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

In [8]:
@torch.no_grad()
def estimate_loss():
        out = {}
        model.eval()
        for split in ["train", "val"]:
            losses = torch.zeros(eval_iters)
            for k in range(eval_iters):
                x, y = get_batch(split)
                _, loss = model(x, y)
                losses[k] = loss.item()
            out[split] = losses.mean().item()
        model.train()
        return out

In [9]:
for step in range(max_iters):

    if step % eval_interval == 0 or step == max_iters - 1:
        losses = estimate_loss()
        print(f"step {step}: train {losses['train']:.4f}, val {losses['val']:.4f}")

    x, y = get_batch("train")                # 1. get data
    logits, loss = model(x, y)               # 2. forward pass
    optimizer.zero_grad(set_to_none=True)    # 3. clear old gradients
    loss.backward()                          # 4. compute new gradients
    optimizer.step()                         # 5. update the table

step 0: train 8.3790, val 8.3780
step 300: train 4.6945, val 4.7029
step 600: train 4.3362, val 4.3673
step 900: train 4.1855, val 4.1868
step 1200: train 4.0717, val 4.0637
step 1500: train 4.0143, val 4.0337
step 1800: train 3.9665, val 3.9412
step 2100: train 3.9256, val 3.9387
step 2400: train 3.8660, val 3.8885
step 2700: train 3.8379, val 3.8538
step 2999: train 3.8597, val 3.8214


In [10]:
from transformers import AutoTokenizer

tok = AutoTokenizer.from_pretrained("tokenizer")
EOT_ID = tok.convert_tokens_to_ids("<|endoftext|>")   # 0

start = torch.tensor([[EOT_ID]], dtype=torch.long, device=device)   # shape (1, 1)
out = model.generate(start, max_new_tokens=100)                     # shape (1, 101)
print(tok.decode(out[0].tolist()))

<|endoftext|>Once upon a time, there was a little boy called Jim really goose. One day, the learning ok. They look at his brother and put it on it the things, played until they put it on the newrel.

One day, they reached every deep him. They put it on it, but Sam is relieved. He is more different anymore and juicy he wrap whenever when Bob are delicate and ate it with theudd gem. It did they ran until Tim managed, sister can take


In [11]:
os.makedirs("model", exist_ok = True)
torch.save(model.state_dict(), 'model/bigram_with_attention.pth')
torch.save(optimizer.state_dict(), 'model/optimizer_bigram_attention.pth')